In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName('Olist Ecommerce Performance Optimization')\
.config('spark.executor.memory', '6g')\
.config('spark.executor.cores', '4')\
.config('spark.executor.instances', '2')\
.config('spark.driver.memory', '4g')\
.config('spark.driver.maxResultSize', '2g')\
.config('spark.sql.shuffle.partitions', '64')\
.config('spark.default.parallelism', '64')\
.config('spark.sql.adaptive.enabled', 'true')\
.config('spark.sql.adaptive.coalescePartition.enabled', 'true')\
.config('spark.sql.autoBroadcastJoinThreshhold', 20*1024*1024)\
.config('spark.sql.files.maxPartitionBytes', '64MB')\
.config('spark.sql.files.openCostInBytes', '2MB')\
.config('spark.memory.fraction', 0.8)\
.config('spark.memory.storageFraction', 0.2)\
.getOrCreate()

26/03/12 22:21:41 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
hdfs_path = '/data/olist/'

customer_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema=True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema=True)
order_items_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv' , header=True, inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv' , header=True, inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv' , header=True, inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',  header=True, inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv' , header=True, inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv' , header=True, inferSchema=True)
product_category_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True, inferSchema=True )

In [4]:
full_orders_df = spark.read.parquet('/data/olist/processed/')

# Optimized Join Strategies

In [4]:
from pyspark.sql import functions as F

In [ ]:
# Broadcasting

customers_broadcast_df = F.broadcast(customer_df)
optimized_broadcast_join = full_orders.join(customers_broadcast_df, 'customer_id')

In [5]:
# Sort and Merge join

sorted_customers_df = customer_df.sortWithinPartitions('customer_id')
sorted_orders_df = full_orders_df.sortWithinPartitions('customer_id')

optimized_merge_full_orders_df = sorted_orders_df.join(sorted_customers_df, 'customer_id')

In [5]:
# Bucket Join

bucketed_customers_df = customer_df.repartition(10,'customer_id')
bucketed_orders_df = full_orders_df.repartition(10,'customer_id')

bucket_join_df = bucketed_orders_df.join(bucketed_customers_df, 'customer_id')

In [ ]:
# Skew Join handling

skew_handled_join = full_orders_df.join(customer_df.hint('skew'), 'customer_id)

# Data Serving Layer
# Objective

* Save and retrieved processed data efficiently inside Dataproc.
* Serve data in a structured way for analysis.
* Use Parquet, Hive and CSV

In [ ]:
# save as Parquet in HDFS

full_orders_df.write.mode('overwrite').parquet('/data/olist/')

In [7]:
# save as a parquet in Goole cloud Storage 

full_orders_df.write.mode('overwrite').parquet('gs://dataproc-staging-us-central1-94103203794-hgjblpcd/temp_data')

In [13]:
# save as table or persistent table

full_orders_df.write.mode('overwrite').saveAsTable('full_order_detail')

26/03/12 22:38:30 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [14]:
spark.sql('show tables').show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|  default|customer_persistent|      false|
|  default|  full_order_detail|      false|
+---------+-------------------+-----------+



In [ ]:
# save as csv in hdfs 

full_orders_df.write.mode('overwrite').option('header', 'true').csv('/data/olist/olist_proc')